# Programming exercise 11: Lindblad master equation and Monte Carlo Wave Function Method

Due on Monday, 06.07.2026, 20h

# The problem

Light propagating through an atomic medium is usually scattered and thus absorbed by near-resonant coupling to a transition from the ground ($|g_1\rangle$) to an excited ($|e\rangle$) electronic state. If a second light field ("coupling laser") couples the excited state to a third, long-lived state ($|g_2\rangle$), the medium can again become transparent for the first light field ("probe laser"). This effect, called electromagnetically induced transparency (EIT), has led to a plethora of applications, see e.g. Rev. Mod. Phys. 77, 633 (2005) [1]. The dynamics of the atomic populations can be simulated using the Lindblad master equation, accounting for the coherent driving laser fields and incoherent processes such as spontaneous emission and dephasing, e.g. due to finite laser linewidth. The complex refractive index of the atomic medium is given by $n=\sqrt{1+\chi}\approx 1 + (\chi_R + \chi_I)/2$, where the real and imaginary part of the complex susceptibility $\chi$ account for diffraction and absorption of the probe light respectively. $\chi$ depends on the polarizability of the atomic dipoles in the medium and is thus proportional to expectation value of the operator $|g_1\rangle \langle e|$, i.e. $\rm{tr}[|g_1\rangle \langle e| \rho] = \langle e| \rho |g_1\rangle = \rho_{eg_1}$. This coherence will thus be the main object of interest.

The interaction with the laser fields is described by the Hamiltonian (in rotating wave approximation, in the basis $\{|g_1\rangle,|g_2\rangle,|e\rangle\}$, following the sign convention used in [1])
$$
H=\left[
\begin{array}{ccc}
0 & 0 & -\Omega_p/2 \\
0 & \Delta_p-\Delta_c & -\Omega_c/2 \\
-\Omega_p/2 & -\Omega_c/2 & \Delta_p
\end{array}
\right]
$$

where $\Omega$'s are Rabi-frequencies and $\Delta$'s are the detunings of the lasers from the respective atomic transitions. $\Delta_p-\Delta_c$ is the "two-photon-detuning" sometimes abbreviated as $\delta_2$.

The dissipative processes we want to consider are the spontaneous emission from $e$ to both stable states $g_1$ and $g_2$ and later also the decay from $g_2$ to $g_1$, which destroys the EIT effect. These are accounted for by the jump operators
$$
\Gamma_{eg_1} = \sqrt{\gamma_p} |g_1\rangle \langle e|, \quad \Gamma_{eg_2} = \sqrt{\gamma_c} |g_2\rangle \langle e|, \quad\Gamma_{g_2g_1} = \sqrt{\gamma_g} |g_1\rangle \langle g_2| .
$$
The resulting master equation reads
$$
\dot\rho = -i[H,\rho] + \mathcal{L}_{\Gamma_{eg_1}} [\rho]+ \mathcal{L}_{\Gamma_{eg_2}} [\rho]+ \mathcal{L}_{\Gamma_{g_2g_1}} [\rho]
$$
with the Lindblad superoperator
$$
\mathcal{L}_{\Gamma_x} [\rho] = \Gamma_x \rho \Gamma_x^\dagger - \frac{1}{2}(\Gamma_x^\dagger\Gamma_x \rho + \rho \Gamma_x^\dagger\Gamma_x)
$$

In [2]:
# load standard libraries

import numpy as np   # standard numerics library
import numpy.linalg as LA

import matplotlib.pyplot as plt   # for making plots

import Comp_Quant_Dynam as cqd 

### Exercise 1: System setup and steady-state calculation

Construct the Liouvillian superoperator in matrix form and test your implementation by calculating the steady state for the case of zero two-photon detuning ($\delta_2=0$) and $\gamma_g=0$, which is the pure state ("dark state")
$$
|d\rangle = \frac{\Omega_c |g_1\rangle - \Omega_p |g_2\rangle}{\sqrt{\Omega_p^2 + \Omega_c^2}}.
$$

Detailed instructions (you can of course proceed differently if you want. I am pretty sure the method described below is not the most elegant one...):
- Building the Liouvillian in matrix form: We want to formulate the master equation in the standard form of an ordinary differential equation $\dot\rho = \mathcal{L} \rho$ with $\rho$ the density matrix in vector form and $\mathcal{L}$ a matrix summarizing the right-hand side of the master equation. However, the master equation is more conveniently formulated using dot-products between matrices representing operators in Hilbert space. Therefore, it makes sense to write a function that takes the vectorized density matrix as an input (along with the Hamiltonian matrix and the jump operators used in the Lindblad terms), converts it into a matrix using reshape(), calculates the right-hand side of the master equation, and returns the result in vector form by using reshape again. To build the Liouvillian matrix you can loop over all the projectors $|i\rangle\langle j|$ and pass them to this function. Those projectors are the basis vectors spanning the space of vectorized density matrices. The outputs will be the columns of the Liouvillian matrix.
- Calculating steady states: The steady state is the unique state for which the right-hand side of the master equation gives zero. Calculating this state numerically is less trivial than it seems. If we just try solving the linear system of equations $\mathcal{L}\rho = 0$ it won't work since $\rho=0$ is always a trivial solution. We need to ensure that the trace of the density matrix is 1. This can be achieved by dropping the first line of the master equation ($\dot\rho_{g_1g_1}=...$) and replacing $\rho_{g_1g_1} = 1-\rho_{g_2g_2}-\rho_{ee}$ everywhere. The resulting inhomogeneous system of 8 equations can be solved using np.linalg.solve(). $\rho_{g_1g_1}$ can be prepended to the solution by using the trace condition again. There are many other ways to calculate steady states; see for example the QuTiP documentation (http://qutip.org/docs/4.0.2/guide/guide-steady.html). You can of course also use QuTiP routines to check your results.

### Exercise 2: Physical properties of the steady state

- Scan the probe detuning $\Delta_p$ at resonant coupling beam ($\Delta_c=0$). Plot imaginary and real part of the steady-state value of $\rho_{eg_1}$ as a function of $\Delta_p$. Start with parameters $\Omega_p=\Omega_c=\gamma_p=1$ and all other parameters zero. You should see the charactersitic EIT dip in the absorption (see Fig. 1 in [1]). Then change the coupling Rabi frequency to a larger value to see what happens (Fig. 7 in [1]). Can you interpret the result in the dressed-state picture? Also change $\gamma_p$ and $\gamma_c$ and describe your observations. In principle, $\gamma_p$ can be set to 1, defining the energy scale.
- Now add a spontaneous decay from $g_2$ to $g_1$. Make the same plots again (imaginary part of the steady state $\rho_{eg_1}$, i.e. absorption, as a function of $\Delta_p$) for a fixed value of $\Omega_c=\gamma_p=1$ and various values of $\gamma_{g}$. What do you observe? Compare to Fig. 10 in [1].

Describe and interpret your results.

### Exercise 3: Dissipative dynamics

Solve the time-dependent master equation with initial state $\rho_0 = |g_1\rangle \langle g_1|$. You can use a scipy integrator, for example the one used in exercise sheet 5. Plot the populations of the levels and the real and imaginary parts of $\rho_{eg_1}$ as a function of time. A good parameter choice is $\Omega_p = 1$, $\Omega_c = \gamma_p = 2$, with all other parameters set to zero, integrating up to $t=20$. Check qualitatively how the relaxation time depends on the parameters, especially on $\Omega_c$ and $\gamma_p$.

### Exercise 4: Monte Carlo wave function method

Now we want to solve the dynamics using the Monte Carlo wave function technique. For such a small system (three-dimensional Hilbert space) the method will not be more efficient than directly integrating the master equation. However, for many-body spin systems, the reduction from a $4^N$- to a $2^N$-dimensional problem, at the expense of stochastic sampling, can be extremely useful.

First, calculate a single jump trajectory by implementing the algorithm discussed in the lecture (see also Rev. Mod. Phys. 70, 101-144, 1998). Initially the system is in state $|g_1\rangle$.

Try the following cases:
- "Standard EIT" case: $\Omega_p=1, \Omega_c=2, \gamma_p=2$ and all other parameters zero. Calculate the evolution up to $t=10$, plotting the populations of the three states. There should not be many jumps in this case. At long times, any single trajectory should go to the correct dark state, which you can check using the analytical solution for the dark state. Even if you do not allow any jumps by setting the jump probability to zero, so that you simply propagate with the non-Hermitian Hamiltonian and renormalize after each step, you already converge to the correct steady state. (Why?)
- "On-off" case. When $\Omega_p$ and $\gamma_p$ are much larger than $\Omega_c$ and $\gamma_c$, we have a situation that is similar to the Dehmelt setup (Phys. Rev. Lett. 57, 1699) with the difference that we have a $\Lambda$-scheme instead of a V-scheme. For the $\Lambda$ scheme we always have EIT, i.e. the system will end up in the dark state after some time. However, if we add a small decay $\gamma_g$, there is no exact dark state any more and we can observe a similar behavior to the "telegraph noise" reported in Phys. Rev. Lett. 57, 1699, where the atom alternates between a fluorescing state and a dark state. Try the parameters $\Omega_p=1, \Omega_c=0.1, \gamma_p=1, \gamma_c=0.1, \gamma_g=0.01$ and integrate to a long time ($t=1000$).
- Optional: In the V-scheme the difference is that the decay is from the $g$-states to the $e$-state. In this case there is no true dark state. On the other hand, the resulting dynamics we get from the quantum jump simulation is not quite what one expects from the experiment with a single ion. This is because the measurement backaction is not included here. If we instead make a ladder scheme with a decay from $g_1$ to $e$ and from $e$ to $g_2$, the telegraph-noise behavior is recovered. Try this case with $\Omega_p=1, \Omega_c=0.1, \bar{\gamma}_p=1, \gamma_c=0.1$, where the overbar indicates that the decay goes into the opposite direction compared to the EIT case. Again integrate to a long time.

Secondly, implement the actual Monte Carlo wave function method: Average the observables over many jump trajectories.

To start, try the parameters of the "Standard EIT" case. For 100 jump trajectories and a time step of 0.1 you should already get reasonable results. Compare the result for the populations and the imaginary part of $\rho_{eg_1}$ (absorption) to the direct integration of the master equation from Exercise 3 above.

Also plot ~50 single trajectories (e.g. plotting $\rho_{g_1g_1}$) into one plot to see what the averaging does.